# Experiment 1.3.6 — Frozen SNN representation/readout ablation

This notebook reuses the completed Experiment 1.3.5 checkpoints and **does not retrain the SNN backbone**.

It probes three questions using fresh train-only `StandardScaler + LogisticRegression` heads:

1. **Layer probe:** raw input count vs L1/L2/L3 spike counts at 250 ms.
2. **Temporal-resolution probe:** frozen L3 spike counts at 250 / 125 / 62.5 ms.
3. **State-readout probe:** L3 spike count vs membrane end-state vs membrane mean at 250 ms.

The trained Experiment 1.3.5 checkpoint is the source of truth for the SNN widths, shifts, sampling rate, membrane time constant, and threshold.

In [ ]:
from __future__ import annotations
from pathlib import Path
import hashlib, math, os, random, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import snntorch as snn
from snntorch import surrogate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from snn.accel_reconstruction_eval.datasets import load_acceleration_data

DATASET_ROOTS = [
    REPO_ROOT / 'outputs/action0_wavelets_0e5_1_2_4_8_sr_64/low-pass/aligned-board-events/segmentation_padded',
    REPO_ROOT / 'outputs/action1_wavelets_0e5_1_2_4_8_sr_64/low-pass/aligned-board-events/segmentation_padded',
]
EVENT_CHANNEL_COUNT = 30
TOTAL_CHANNEL_COUNT = 36
EXPECTED_FS = 64.0
INCLUDED_LABELS = ('A','B','C','D','E','X','G','H','I','J','K','L')
SPLIT_SEED = 12345
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
SNN_SEEDS = (11, 23, 101)
COARSE_BIN_MS = 250.0
SURROGATE_SLOPE = 25.0
RESET_MECHANISM = 'subtract'
BATCH_SIZE = 64
NUM_WORKERS = 0
LOGREG_MAX_ITER = 5000
LOGREG_C = 1.0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SOURCE_EXPERIMENT_ID = 'experiment_1_3_5_continuous_snn_local_features'
SOURCE_RESULTS_DIR = REPO_ROOT / 'notebooks/artifacts' / SOURCE_EXPERIMENT_ID
EXPERIMENT_ID = 'experiment_1_3_6_snn_representation_readout_ablation'
RESULTS_DIR = REPO_ROOT / 'notebooks/artifacts' / EXPERIMENT_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def derive_seed(master_seed: int, *parts: object) -> int:
    text = '|'.join([str(master_seed), *(str(p) for p in parts)])
    return int.from_bytes(hashlib.sha256(text.encode()).digest()[:4], 'little')


def shift_to_alpha(shift: int) -> float:
    return float(1.0 - 2.0 ** (-int(shift)))


def allocate_neurons(width: int, shifts: tuple[int, ...]) -> tuple[int, ...]:
    base, remainder = divmod(width, len(shifts))
    counts = [base] * len(shifts)
    order = []
    left, right = 0, len(shifts) - 1
    while left <= right:
        order.append(left)
        if left != right:
            order.append(right)
        left += 1; right -= 1
    for i in range(remainder):
        counts[order[i]] += 1
    return tuple(counts)


def build_alpha_tensor(width: int, shifts: tuple[int, ...]) -> torch.Tensor:
    vals = []
    for shift, count in zip(shifts, allocate_neurons(width, shifts), strict=True):
        vals.extend([shift_to_alpha(shift)] * count)
    return torch.tensor(vals, dtype=torch.float32)

print('Repository root:', REPO_ROOT)
print('Device:', DEVICE)
print('Source results:', SOURCE_RESULTS_DIR)

In [ ]:
# Load the same cohort and reconstruct the exact fixed user-disjoint split.
data = load_acceleration_data(DATASET_ROOTS, repository_root=REPO_ROOT, require_reconstruction=False)
fs_values = {float(m.sampling_rate_hz) for m in data.producer_metadatas}
if len(fs_values) != 1:
    raise ValueError(f'Expected one shared sampling rate, got {fs_values}')
SAMPLING_RATE_HZ = fs_values.pop()
if not np.isclose(SAMPLING_RATE_HZ, EXPECTED_FS):
    raise ValueError((SAMPLING_RATE_HZ, EXPECTED_FS))

rows = []
keep = set(INCLUDED_LABELS)
for package_index, package in enumerate(data.packages):
    for segment_index, label in enumerate(package.labels.astype(str)):
        if label in keep:
            rows.append(dict(
                package_index=package_index,
                segment_index=segment_index,
                user=str(package.user), action=str(package.action), label=str(label),
                valid_length=int(package.valid_lengths[segment_index]),
                package_padded_length=int(package.padded_spike_imu.shape[1]),
                sample_id=f'{package.user}/action_{package.action}/{segment_index}',
            ))
manifest = pd.DataFrame(rows)
labels_sorted = sorted(manifest.label.unique().tolist())
CLASS_TO_IDX = {lab: i for i, lab in enumerate(labels_sorted)}
manifest['label_idx'] = manifest.label.map(CLASS_TO_IDX).astype(int)
N_CLASSES = len(labels_sorted)
GLOBAL_PADDED_LENGTH = int(manifest.package_padded_length.max())
COARSE_BIN_SAMPLES = int(np.rint(COARSE_BIN_MS * SAMPLING_RATE_HZ / 1000.0))
N_COARSE_BINS = int(math.ceil(GLOBAL_PADDED_LENGTH / COARSE_BIN_SAMPLES))
PADDED_LENGTH = N_COARSE_BINS * COARSE_BIN_SAMPLES


def make_user_split(split_seed: int):
    users = np.asarray(sorted(manifest.user.unique()), dtype=object)
    rng = np.random.default_rng(derive_seed(split_seed, 'user_split'))
    rng.shuffle(users)
    n_train = int(round(TRAIN_FRACTION * len(users)))
    n_val = int(round(VAL_FRACTION * len(users)))
    train_users = set(users[:n_train]); val_users = set(users[n_train:n_train+n_val]); test_users = set(users[n_train+n_val:])
    part = lambda us: manifest[manifest.user.isin(us)].reset_index(drop=True)
    return part(train_users), part(val_users), part(test_users)


def build_events(df: pd.DataFrame) -> np.ndarray:
    out = np.zeros((len(df), PADDED_LENGTH, EVENT_CHANNEL_COUNT), np.float32)
    for i, row in enumerate(df.itertuples(index=False)):
        p = data.packages[int(row.package_index)]
        x = np.asarray(p.padded_spike_imu[int(row.segment_index), :, :EVENT_CHANNEL_COUNT], np.float32)
        out[i, :min(len(x), PADDED_LENGTH)] = x[:PADDED_LENGTH]
    return out

train_df, val_df, test_df = make_user_split(SPLIT_SEED)
X_train, X_val, X_test = map(build_events, (train_df, val_df, test_df))
y_train = train_df.label_idx.to_numpy(np.int64); y_val = val_df.label_idx.to_numpy(np.int64); y_test = test_df.label_idx.to_numpy(np.int64)
print('split shapes:', X_train.shape, X_val.shape, X_test.shape)
print('coarse bins:', N_COARSE_BINS, 'samples/bin:', COARSE_BIN_SAMPLES)

In [ ]:
# Read the ACTUAL Experiment 1.3.5 configuration from the saved checkpoints.
def checkpoint_path(seed: int) -> Path:
    return SOURCE_RESULTS_DIR / f'snn_seed_{seed}.pt'


def canonical_config(payload: dict) -> dict:
    cfg = payload.get('config')
    required = ('experiment_id','fs','bin_samples','widths','shifts','tau_mem_ms','threshold','split_seed')
    if cfg is None or any(k not in cfg for k in required):
        raise ValueError(f'Unexpected 1.3.5 checkpoint config: {cfg}')
    return dict(
        experiment_id=str(cfg['experiment_id']), fs=float(cfg['fs']), bin_samples=int(cfg['bin_samples']),
        widths=tuple(int(v) for v in cfg['widths']),
        shifts=tuple(tuple(int(s) for s in layer) for layer in cfg['shifts']),
        tau_mem_ms=float(cfg['tau_mem_ms']), threshold=float(cfg['threshold']), split_seed=int(cfg['split_seed']),
    )

reference_payload = torch.load(checkpoint_path(SNN_SEEDS[0]), map_location='cpu', weights_only=False)
REFERENCE_CONFIG = canonical_config(reference_payload)
if REFERENCE_CONFIG['experiment_id'] != SOURCE_EXPERIMENT_ID:
    raise ValueError(REFERENCE_CONFIG)
if not np.isclose(REFERENCE_CONFIG['fs'], SAMPLING_RATE_HZ):
    raise ValueError('Checkpoint/data sampling-rate mismatch')
if REFERENCE_CONFIG['bin_samples'] != COARSE_BIN_SAMPLES or REFERENCE_CONFIG['split_seed'] != SPLIT_SEED:
    raise ValueError('Checkpoint/data protocol mismatch')

SNN_LAYER_WIDTHS = REFERENCE_CONFIG['widths']
SNN_LAYER_SHIFTS = REFERENCE_CONFIG['shifts']
TAU_MEM_MS = REFERENCE_CONFIG['tau_mem_ms']
THRESHOLD = REFERENCE_CONFIG['threshold']
BETA = float(math.exp(-(1000.0 / SAMPLING_RATE_HZ) / TAU_MEM_MS))

class ContinuousLocalFeatureSNN(nn.Module):
    def __init__(self):
        super().__init__()
        h1, h2, d = SNN_LAYER_WIDTHS
        grad = surrogate.fast_sigmoid(slope=SURROGATE_SLOPE)
        self.fc1 = nn.Linear(EVENT_CHANNEL_COUNT, h1, bias=False)
        self.lif1 = snn.Synaptic(alpha=build_alpha_tensor(h1, SNN_LAYER_SHIFTS[0]), beta=BETA, threshold=THRESHOLD, spike_grad=grad, reset_mechanism=RESET_MECHANISM)
        self.fc2 = nn.Linear(h1, h2, bias=False)
        self.lif2 = snn.Synaptic(alpha=build_alpha_tensor(h2, SNN_LAYER_SHIFTS[1]), beta=BETA, threshold=THRESHOLD, spike_grad=grad, reset_mechanism=RESET_MECHANISM)
        self.fc3 = nn.Linear(h2, d, bias=False)
        self.lif3 = snn.Synaptic(alpha=build_alpha_tensor(d, SNN_LAYER_SHIFTS[2]), beta=BETA, threshold=THRESHOLD, spike_grad=grad, reset_mechanism=RESET_MECHANISM)
        self.classifier = nn.Linear(N_COARSE_BINS * d, N_CLASSES, bias=True)


def load_frozen_checkpoint(seed: int):
    payload = torch.load(checkpoint_path(seed), map_location='cpu', weights_only=False)
    got = canonical_config(payload)
    if got != REFERENCE_CONFIG:
        raise ValueError(f'Checkpoint config mismatch for seed {seed}: {got} vs {REFERENCE_CONFIG}')
    model = ContinuousLocalFeatureSNN()
    model.load_state_dict(payload['model_state_dict'], strict=True)
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    return model, payload

# Read uploaded 1.3.5 results and cross-check the three checkpoint metrics.
source_results = pd.read_csv(SOURCE_RESULTS_DIR / 'results.csv')
checkpoint_rows = []
for seed in SNN_SEEDS:
    _, payload = load_frozen_checkpoint(seed)
    csv_row = source_results[(source_results.model == 'continuous_snn_plus_linear') & (source_results.seed == float(seed))].iloc[0]
    if not np.isclose(payload['test']['balanced_accuracy'], csv_row.test_balanced_accuracy):
        raise ValueError(f'Checkpoint/results.csv mismatch for seed {seed}')
    checkpoint_rows.append(dict(seed=seed, best_epoch=payload['best_epoch'], val_BA=payload['val']['balanced_accuracy'], test_BA_1_3_5=payload['test']['balanced_accuracy']))

print('Frozen architecture:', SNN_LAYER_WIDTHS, SNN_LAYER_SHIFTS)
display(pd.DataFrame(checkpoint_rows))
display(source_results)

In [ ]:
# Efficient frozen feature extraction: accumulate only the representations needed by the probes.
@torch.no_grad()
def extract_features(model: ContinuousLocalFeatureSNN, X: np.ndarray) -> dict[str, np.ndarray]:
    model = model.to(DEVICE); model.eval()
    h1, h2, d = SNN_LAYER_WIDTHS
    outputs = {k: [] for k in ('l1_250','l2_250','l3_250','l3_125','l3_62_5','l3_mem_end_250','l3_mem_mean_250')}
    loader = DataLoader(TensorDataset(torch.from_numpy(X)), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    for (xb,) in loader:
        xb = xb.to(DEVICE, non_blocking=True); B, T, _ = xb.shape
        syn1=torch.zeros(B,h1,device=DEVICE); mem1=torch.zeros_like(syn1)
        syn2=torch.zeros(B,h2,device=DEVICE); mem2=torch.zeros_like(syn2)
        syn3=torch.zeros(B,d,device=DEVICE); mem3=torch.zeros_like(syn3)
        acc1=torch.zeros(B,h1,device=DEVICE); acc2=torch.zeros(B,h2,device=DEVICE); acc3=torch.zeros(B,d,device=DEVICE)
        acc125=torch.zeros(B,d,device=DEVICE); acc62=torch.zeros(B,d,device=DEVICE); mem_sum=torch.zeros(B,d,device=DEVICE)
        f1=[]; f2=[]; f3=[]; f125=[]; f62=[]; mend=[]; mmean=[]
        for t in range(T):
            s1,syn1,mem1=model.lif1(model.fc1(xb[:,t]),syn1,mem1)
            s2,syn2,mem2=model.lif2(model.fc2(s1),syn2,mem2)
            s3,syn3,mem3=model.lif3(model.fc3(s2),syn3,mem3)
            acc1 += s1; acc2 += s2; acc3 += s3; acc125 += s3; acc62 += s3; mem_sum += mem3
            if (t+1) % 4 == 0:
                f62.append(acc62.clone()); acc62.zero_()
            if (t+1) % 8 == 0:
                f125.append(acc125.clone()); acc125.zero_()
            if (t+1) % COARSE_BIN_SAMPLES == 0:
                f1.append(acc1.clone()); f2.append(acc2.clone()); f3.append(acc3.clone())
                mend.append(mem3.clone()); mmean.append(mem_sum / COARSE_BIN_SAMPLES)
                acc1.zero_(); acc2.zero_(); acc3.zero_(); mem_sum.zero_()
        batch = {
            'l1_250': torch.stack(f1,1), 'l2_250': torch.stack(f2,1), 'l3_250': torch.stack(f3,1),
            'l3_125': torch.stack(f125,1), 'l3_62_5': torch.stack(f62,1),
            'l3_mem_end_250': torch.stack(mend,1), 'l3_mem_mean_250': torch.stack(mmean,1),
        }
        for k,v in batch.items(): outputs[k].append(v.cpu().numpy().astype(np.float32))
    return {k: np.concatenate(v, axis=0) for k,v in outputs.items()}


def count_bins(X: np.ndarray, samples_per_bin: int) -> np.ndarray:
    N,T,D = X.shape
    return X.reshape(N, T//samples_per_bin, samples_per_bin, D).sum(2)


def metrics(y, p):
    return dict(accuracy=float(accuracy_score(y,p)), balanced_accuracy=float(balanced_accuracy_score(y,p)), macro_f1=float(f1_score(y,p,average='macro')))


def fit_probe(Ftr,Fva,Fte,tag):
    A=Ftr.reshape(len(Ftr),-1); B=Fva.reshape(len(Fva),-1); C=Fte.reshape(len(Fte),-1)
    scaler=StandardScaler(); A=scaler.fit_transform(A); B=scaler.transform(B); C=scaler.transform(C)
    clf=LogisticRegression(max_iter=LOGREG_MAX_ITER,solver='lbfgs',C=LOGREG_C,random_state=derive_seed(SPLIT_SEED,'probe',tag))
    clf.fit(A,y_train)
    return dict(feature_dim=A.shape[1], train=metrics(y_train,clf.predict(A)), val=metrics(y_val,clf.predict(B)), test=metrics(y_test,clf.predict(C)))

raw250 = (count_bins(X_train,COARSE_BIN_SAMPLES), count_bins(X_val,COARSE_BIN_SAMPLES), count_bins(X_test,COARSE_BIN_SAMPLES))
raw_probe = fit_probe(*raw250, tag='raw250')
print('Raw 250 ms test BA:', raw_probe['test']['balanced_accuracy'])

In [ ]:
probe_rows=[]; representation_rows=[]
for split in ('train','val','test'):
    probe_rows.append(dict(group='A_layer_probe',representation='raw_input_250ms_count',snn_seed=np.nan,split=split,feature_dim=raw_probe['feature_dim'],**raw_probe[split]))

for seed in SNN_SEEDS:
    print('='*90); print('Frozen seed',seed)
    model,_=load_frozen_checkpoint(seed)
    print('  extracting train...'); tr=extract_features(model,X_train)
    print('  extracting val...'); va=extract_features(model,X_val)
    print('  extracting test...'); te=extract_features(model,X_test)
    groups = {
        'A_layer_probe': [('l1_spike_250ms_count','l1_250'),('l2_spike_250ms_count','l2_250'),('l3_spike_250ms_count','l3_250')],
        'B_temporal_resolution': [('l3_spike_250ms_count','l3_250'),('l3_spike_125ms_count','l3_125'),('l3_spike_62.5ms_count','l3_62_5')],
        'C_state_readout': [('l3_spike_250ms_count','l3_250'),('l3_mem_250ms_end','l3_mem_end_250'),('l3_mem_250ms_mean','l3_mem_mean_250')],
    }
    for group,specs in groups.items():
        for rep,key in specs:
            result=fit_probe(tr[key],va[key],te[key],tag=(seed,group,rep))
            representation_rows.append(dict(group=group,snn_seed=seed,representation=rep,shape_per_sample=str(tr[key].shape[1:]),feature_dim=result['feature_dim']))
            for split in ('train','val','test'):
                probe_rows.append(dict(group=group,representation=rep,snn_seed=seed,split=split,feature_dim=result['feature_dim'],**result[split]))
    del tr,va,te,model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

probe_df=pd.DataFrame(probe_rows)
representation_df=pd.DataFrame(representation_rows).drop_duplicates().reset_index(drop=True)
print('probe rows:',len(probe_df)); display(representation_df)

In [ ]:
def summarize(group, order, include_raw=False):
    d=probe_df.query("group == @group and split == 'test'").copy()
    d=d[d.snn_seed.notna()]
    rows=[]
    if include_raw:
        r=probe_df.query("group == 'A_layer_probe' and representation == 'raw_input_250ms_count' and split == 'test'").iloc[0]
        rows.append(dict(representation='raw_input_250ms_count',n_runs=1,mean_test_BA=r.balanced_accuracy,sd_test_BA=np.nan,mean_test_accuracy=r.accuracy,mean_test_macro_f1=r.macro_f1,feature_dim=int(r.feature_dim)))
    for rep in order:
        g=d[d.representation==rep]
        rows.append(dict(representation=rep,n_runs=len(g),mean_test_BA=g.balanced_accuracy.mean(),sd_test_BA=g.balanced_accuracy.std(ddof=1),mean_test_accuracy=g.accuracy.mean(),mean_test_macro_f1=g.macro_f1.mean(),feature_dim=int(g.feature_dim.iloc[0])))
    return pd.DataFrame(rows)

layer_order=['l1_spike_250ms_count','l2_spike_250ms_count','l3_spike_250ms_count']
temporal_order=['l3_spike_250ms_count','l3_spike_125ms_count','l3_spike_62.5ms_count']
state_order=['l3_spike_250ms_count','l3_mem_250ms_end','l3_mem_250ms_mean']
layer_summary=summarize('A_layer_probe',layer_order,include_raw=True)
temporal_summary=summarize('B_temporal_resolution',temporal_order)
state_summary=summarize('C_state_readout',state_order)
display(layer_summary); display(temporal_summary); display(state_summary)

for title,df,labels in [
    ('1.3.6A — Layer probe',layer_summary,['Raw','L1','L2','L3']),
    ('1.3.6B — L3 temporal resolution',temporal_summary,['250 ms','125 ms','62.5 ms']),
    ('1.3.6C — L3 readout type',state_summary,['Spike count','Mem end','Mem mean']),
]:
    plt.figure(figsize=(8,4.5)); x=np.arange(len(df)); plt.bar(x,df.mean_test_BA,yerr=df.sd_test_BA.fillna(0),capsize=4)
    plt.xticks(x,labels); plt.ylabel('Test balanced accuracy'); plt.title(title); plt.grid(axis='y',alpha=.25); plt.show()

probe_df.to_csv(RESULTS_DIR/'all_probe_results.csv',index=False)
representation_df.to_csv(RESULTS_DIR/'representation_dimensions.csv',index=False)
layer_summary.to_csv(RESULTS_DIR/'layer_probe_summary.csv',index=False)
temporal_summary.to_csv(RESULTS_DIR/'temporal_resolution_summary.csv',index=False)
state_summary.to_csv(RESULTS_DIR/'state_readout_summary.csv',index=False)
print('Saved to',RESULTS_DIR)